# ДЗ 4. Операции, axis и broadcasting на данных «Дала Маркет»
Срок: **до понедельника 16.11, 19:00**.
Сдача: папка `hw04` репозитория `dsa-homework` (`hw04.ipynb`): `git add hw04` → `git commit` → `git push`, ссылку прикрепите в LMS.

Каждый ответ: **число + одна фраза для менеджера**. Циклы `for` по заказам не используем. Небольшой цикл по городам или категориям при сборке таблицы допустим.

- **[База]** обязательно для всех · **[Стандарт]** сколько получится · **[Челлендж]** для тех, кому легко.

## [Для всех] Чтение кода: без запуска
Сначала напишите ответ и «почему», потом запустите.
```python
import numpy as np
# 1
a = np.array([2, 4, 6]); print(a / 2, a // 4)
# 2
m = np.ones((3, 4)); print(m.sum(axis=1))
# 3
print(np.arange(3) + np.arange(3).reshape(3, 1))
# 4
x = np.array([1, 4, 9]); print(np.sqrt(x).sum())
# 5
print((np.arange(1, 5) ** 2).mean())
```

Ваши ответы и объяснения:

1. `[1. 2. 3.] [0 1 1]`: `/` делит с получением float, `//` выполняет целочисленное деление.
2. `[4. 4. 4.]`: `axis=1` суммирует значения в каждой строке; в каждой строке четыре единицы.
3. Получится матрица `[[0, 1, 2], [1, 2, 3], [2, 3, 4]]`: broadcasting складывает вектор с каждым элементом столбца.
4. `6.0`: квадратные корни равны 1, 2 и 3, их сумма — 6.
5. `7.5`: среднее квадратов чисел 1, 2, 3, 4 равно (1+4+9+16)/4.


In [1]:
# Загрузка orders.csv: в Colab нажмите «Выбрать файлы» и укажите orders.csv из LMS.
# Если файл уже лежит рядом с ноутбуком, загрузка пропускается.
import os
if not os.path.exists("orders.csv"):
    try:
        from google.colab import files
        files.upload()
    except ImportError:
        print("Положите orders.csv в одну папку с ноутбуком")
print("orders.csv на месте:", os.path.exists("orders.csv"))

Saving orders.csv to orders (1).csv
orders.csv на месте: True


In [2]:
import csv
import numpy as np

CITY = {"алматы": "Алматы", "almaty": "Алматы", "астана": "Астана", "astana": "Астана",
        "шымкент": "Шымкент", "shymkent": "Шымкент", "караганда": "Караганда",
        "актобе": "Актобе", "атырау": "Атырау"}

with open("orders.csv", encoding="utf-8") as f:
    rows = list(csv.DictReader(f))

amount = np.array([float(r["amount"].replace(" ", "")) for r in rows])
paid   = np.array([r["status"] == "оплачен" for r in rows])
days   = np.array([float(r["delivery_days"]) if r["delivery_days"] else np.nan for r in rows])
city   = np.array([CITY.get(r["city"].strip().lower(), r["city"].strip()) for r in rows])
cust   = np.array([r["customer_id"] for r in rows])
month  = np.array([r["order_date"][:7] for r in rows])      # "2026-09" или "2026-10"
cat    = np.array([r["category"] for r in rows])
oid    = np.array([r["order_id"] for r in rows])

print(amount.shape, np.unique(month))

(5463,) ['2026-09' '2026-10']


In [3]:
# Чистые данные: без дублей (оставляем первое вхождение order_id), без TEST, только оплаченные
_, first = np.unique(oid, return_index=True)
keep = np.zeros(len(oid), dtype=bool)
keep[first] = True
clean = keep & (cust != "TEST") & paid
print("строк всего:", len(oid), "| в чистых данных:", clean.sum())

строк всего: 5463 | в чистых данных: 4739


## [База] 1. Выручка по чистым данным
Сколько строк осталось в `clean`? Выручка за сентябрь и октябрь и изменение в %. Совпадает ли с ДЗ 2?

In [4]:

months_order = np.array(["2026-09", "2026-10"])
clean_count = clean.sum()
revenue_by_month = np.array([
    amount[clean & (month == m)].sum() for m in months_order
])
sep_revenue, oct_revenue = revenue_by_month
change_pct = ((oct_revenue - sep_revenue) / sep_revenue * 100
              if sep_revenue != 0 else np.nan)

print("1) Выручка по чистым данным")
print("Строк в clean:", clean_count)
print("Сентябрь:", f"{sep_revenue:,.2f} ₸")
print("Октябрь:", f"{oct_revenue:,.2f} ₸")
print("Изменение:", f"{change_pct:.2f}%" if np.isfinite(change_pct) else "нельзя вычислить: выручка сентября равна 0")
print("Комментарий: сравни эти значения с ДЗ 2; небольшое расхождение возможно из-за очистки дублей и TEST.")


1) Выручка по чистым данным
Строк в clean: 4739
Сентябрь: 35,243,410.00 ₸
Октябрь: 34,419,520.00 ₸
Изменение: -2.34%
Комментарий: сравни эти значения с ДЗ 2; небольшое расхождение возможно из-за очистки дублей и TEST.


## [База] 2. Таблица «категория × месяц»
Соберите таблицу 5 × 2 по образцу таблицы городов с урока (категории: `np.unique(cat)`).
Какая категория приносит больше всего выручки за два месяца (`axis`)? Какая упала сильнее всех в %?

In [5]:

categories = np.unique(cat)
table_cat = np.zeros((len(categories), len(months_order)), dtype=float)

for i, category in enumerate(categories):
    for j, m in enumerate(months_order):
        mask = clean & (cat == category) & (month == m)
        table_cat[i, j] = amount[mask].sum()

print("2) Выручка по категориям (строки — категории, столбцы — сентябрь/октябрь)")
print("Категории:", categories)
print(table_cat)

category_total = table_cat.sum(axis=1)
i_top = np.argmax(category_total)
print("Больше всего выручки за два месяца:", categories[i_top],
      f"({category_total[i_top]:,.2f} ₸)")

drop_pct = np.where(
    table_cat[:, 0] != 0,
    (table_cat[:, 1] - table_cat[:, 0]) / table_cat[:, 0] * 100,
    np.nan
)
if np.any(np.isfinite(drop_pct)):
    i_drop = np.nanargmin(drop_pct)
    print("Сильнее всего упала в процентах:", categories[i_drop],
          f"({drop_pct[i_drop]:.2f}%)")
else:
    print("Процентное изменение нельзя вычислить: нет ненулевой выручки сентября.")
print("Комментарий: сравниваем сумму выручки категорий и процентное изменение от сентября к октябрю.")


2) Выручка по категориям (строки — категории, столбцы — сентябрь/октябрь)
Категории: ['Дом' 'Книги' 'Красота' 'Одежда' 'Электроника']
[[ 4997730.  5125790.]
 [ 1065300.  1140910.]
 [ 2610950.  2678990.]
 [ 9167100.  8505730.]
 [17402330. 16968100.]]
Больше всего выручки за два месяца: Электроника (34,370,430.00 ₸)
Сильнее всего упала в процентах: Одежда (-7.21%)
Комментарий: сравниваем сумму выручки категорий и процентное изменение от сентября к октябрю.


## [База] 3. Средний чек по категориям в октябре
Соберите вторую таблицу: **число** заказов по категориям и месяцам (сумма маски, а не сумма amount).
Средний чек = выручка / число заказов, одной операцией над двумя таблицами. У какой категории он самый высокий?

In [6]:

count_table = np.zeros((len(categories), len(months_order)), dtype=int)

for i, category in enumerate(categories):
    for j, m in enumerate(months_order):
        mask = clean & (cat == category) & (month == m)
        count_table[i, j] = mask.sum()

avg_check_table = np.divide(
    table_cat, count_table,
    out=np.full(table_cat.shape, np.nan, dtype=float),
    where=count_table != 0
)

print("3) Количество заказов по категориям (сентябрь, октябрь)")
print(count_table)
print("Средний чек по категориям (сентябрь, октябрь)")
print(avg_check_table)

oct_avg = avg_check_table[:, 1]
if np.any(np.isfinite(oct_avg)):
    i_avg = np.nanargmax(oct_avg)
    print("Самый высокий средний чек в октябре:", categories[i_avg],
          f"({oct_avg[i_avg]:,.2f} ₸)")
else:
    print("Нет октябрьских заказов для расчёта среднего чека.")
print("Комментарий: средний чек — это выручка категории, делённая на число её заказов.")


3) Количество заказов по категориям (сентябрь, октябрь)
[[476 476]
 [265 299]
 [422 435]
 [678 652]
 [531 505]]
Средний чек по категориям (сентябрь, октябрь)
[[10499.43277311 10768.46638655]
 [ 4020.          3815.75250836]
 [ 6187.08530806  6158.59770115]
 [13520.79646018 13045.59815951]
 [32772.74952919 33600.1980198 ]]
Самый высокий средний чек в октябре: Электроника (33,600.20 ₸)
Комментарий: средний чек — это выручка категории, делённая на число её заказов.


## [Стандарт] 4. Доля городов в выручке месяца
Используйте `table` городов с урока. Доля каждого города в выручке своего месяца: `table / table.sum(axis=0)`.
Проверьте, что каждый столбец в сумме даёт 1. Как изменилась доля Шымкента?

In [7]:
cities = np.unique(city)
table = np.zeros((len(cities), len(months_order)), dtype=float)

for i, c in enumerate(cities):
    for j, m in enumerate(months_order):
        mask = clean & (city == c) & (month == m)
        table[i, j] = amount[mask].sum()

month_totals = table.sum(axis=0)
share_by_city = np.divide(
    table, month_totals,
    out=np.zeros_like(table),
    where=month_totals != 0
)

print("4) Доля каждого города в выручке месяца")
print("Города:", cities)
print("Таблица выручки (сентябрь, октябрь):")
print(table)
print("Доли по городам (сентябрь, октябрь):")
print(share_by_city)
print("Сумма долей по каждому месяцу:", share_by_city.sum(axis=0))

sh_idx = np.where(cities == "Шымкент")[0]
if len(sh_idx):
    k = sh_idx[0]
    print("Доля Шымкента: сентябрь", f"{share_by_city[k, 0]:.2%}",
          "| октябрь", f"{share_by_city[k, 1]:.2%}",
          "| изменение, п.п.", f"{(share_by_city[k, 1] - share_by_city[k, 0]) * 100:.2f}")
else:
    print("Шымкент не найден в данных.")
print("Комментарий: доля показывает, какую часть выручки месяца обеспечивает каждый город.")


4) Доля каждого города в выручке месяца
Города: ['Актобе' 'Алматы' 'Астана' 'Атырау' 'Караганда' 'Шымкент']
Таблица выручки (сентябрь, октябрь):
[[ 2576960.  2540420.]
 [14501070. 14643020.]
 [ 9034220.  8850530.]
 [  923990.   969860.]
 [ 3648520.  3371680.]
 [ 4558650.  4044010.]]
Доли по городам (сентябрь, октябрь):
[[0.07311892 0.07380754]
 [0.41145479 0.42542778]
 [0.25633785 0.25713694]
 [0.02621738 0.02817762]
 [0.10352347 0.09795837]
 [0.12934759 0.11749176]]
Сумма долей по каждому месяцу: [1. 1.]
Доля Шымкента: сентябрь 12.93% | октябрь 11.75% | изменение, п.п. -1.19
Комментарий: доля показывает, какую часть выручки месяца обеспечивает каждый город.


## [Стандарт] 5. Кто дал падение
Вклад каждого города в общее падение выручки: изменение города / общее изменение. Сколько процентов падения даёт Шымкент?
Напишите менеджеру одну фразу: где искать причину.

In [8]:
city_change = table[:, 1] - table[:, 0]
total_change = table[:, 1].sum() - table[:, 0].sum()

if total_change != 0:
    contribution_pct = city_change / total_change * 100
else:
    contribution_pct = np.full(len(cities), np.nan)

print("5) Вклад городов в общее изменение выручки")
for i, c in enumerate(cities):
    print(f"{c}: изменение {city_change[i]:,.2f} ₸; вклад {contribution_pct[i]:.2f}%")

sh_idx = np.where(cities == "Шымкент")[0]
if len(sh_idx) and total_change != 0:
    k = sh_idx[0]
    print("Вклад Шымкента в общее падение:", f"{contribution_pct[k]:.2f}%")
else:
    print("Вклад Шымкента не удалось вычислить.")

print("Фраза менеджеру: сначала проверьте города с наибольшим положительным вкладом в падение — там снижение выручки сильнее всего.")


5) Вклад городов в общее изменение выручки
Актобе: изменение -36,540.00 ₸; вклад 4.44%
Алматы: изменение 141,950.00 ₸; вклад -17.23%
Астана: изменение -183,690.00 ₸; вклад 22.30%
Атырау: изменение 45,870.00 ₸; вклад -5.57%
Караганда: изменение -276,840.00 ₸; вклад 33.60%
Шымкент: изменение -514,640.00 ₸; вклад 62.46%
Вклад Шымкента в общее падение: 62.46%
Фраза менеджеру: сначала проверьте города с наибольшим положительным вкладом в падение — там снижение выручки сильнее всего.


## [Стандарт] 6. Исправьте код от AI
AI написал код «доля сентября и октября внутри каждого города». Код падает. Объясните, почему, и исправьте.
```python
share = table / table.sum(axis=1)
```

In [9]:
# Ошибка исходного AI-кода: table.sum(axis=1) имеет форму (число городов,),
# а для деления матрицы (города × месяцы) нужно добавить ось-столбец
# с помощью keepdims=True.
share_inside_city = np.divide(
    table,
    table.sum(axis=1, keepdims=True),
    out=np.zeros_like(table),
    where=table.sum(axis=1, keepdims=True) != 0
)

print("6) Доля сентября и октября внутри каждого города")
for i, c in enumerate(cities):
    print(f"{c}: сентябрь {share_inside_city[i, 0]:.2%}; "
          f"октябрь {share_inside_city[i, 1]:.2%}")
print("Проверка — сумма долей по строкам:", share_inside_city.sum(axis=1))
print("Комментарий: keepdims=True сохраняет форму (число городов, 1), поэтому broadcasting работает по строкам.")


6) Доля сентября и октября внутри каждого города
Актобе: сентябрь 50.36%; октябрь 49.64%
Алматы: сентябрь 49.76%; октябрь 50.24%
Астана: сентябрь 50.51%; октябрь 49.49%
Атырау: сентябрь 48.79%; октябрь 51.21%
Караганда: сентябрь 51.97%; октябрь 48.03%
Шымкент: сентябрь 52.99%; октябрь 47.01%
Проверка — сумма долей по строкам: [1. 1. 1. 1. 1. 1.]
Комментарий: keepdims=True сохраняет форму (число городов, 1), поэтому broadcasting работает по строкам.
